In [1]:
import numpy as np
from scipy.stats import chi2, binom

def est_T(frecuencias: list[int], probabilidades: list[float], n: int, K: int):
    t = 0
    for k in range(K):
        t = t + (frecuencias[k] - n * probabilidades[k]
                 )**2 / (n * probabilidades[k])

    return t

### Ejercicio 5.
Calcular una aproximación del *p−valor* de la prueba de que los siguientes datos corresponden a una distribución binomial con parámetros ($n = 8, p$), donde $p$ no se conoce:
$$ 6, 7, 3, 4, 7, 3, 7, 2, 6, 3, 7, 8, 2, 1, 3, 5, 8, 7 $$

---

#### Aproximación mediante prueba de Pearson
Tenemos $H_0: \text{los datos observados provienen de una distribución Bn(n=8, p)}$, donde p es desconocido.

Como sabemos que si $X \sim Bn(n, p)$, entonces $E[X] = pn$, por lo cual podemos estimar p mediante la media muestral.
$$ \hat{p} = \frac{\overline{X}(m)}{n} = \frac{\overline{X}(18)}{8}, $$
Siendo m la cantidad de datos.

Una vez estimado $\hat p$, tenemos  una distribución $\hat F = Bin(n, \hat p)$, y las probabilidades teóricas son
$$
\hat p_k = P_{\hat F}(X=k) = \binom{8}{k} \ \hat p^k \ (1-\hat p)^{8-k},
\qquad k=0,\ldots,8.
$$

In [2]:
datos = [6, 7, 3, 4, 7, 3, 7, 2, 6, 3, 7, 8, 2, 1, 3, 5, 8, 7]
m = len(datos)
n = 8

p_est = np.mean(datos) / n

# frecuencia de 0 hasta el 8 en los datos
frecuencias = [datos.count(k) for k in range(n + 1)]

# calculando las probabilidades utilizando la fun masa para los valores de 0 a 8
probabilidades = [binom.pmf(k, n, p_est) for k in range(n + 1)]

K = n + 1 # 0, 1, ..., 7, 8
n = sum(frecuencias)

t_obs = est_T(frecuencias, probabilidades, n, K)

# los grados de libertad son K - 2, ya que se esta estimando 1 parámetro (p)
p_valor = chi2.sf(t_obs, df=(K - 2))

print("p_est: ", p_est)
print("t_obs: ", t_obs)
print("p_valor: ", p_valor)

p_est:  0.6180555555555556
t_obs:  31.499330934155314
p_valor:  5.027994320424104e-05


Como *p-valor* es muy pequeño, se rechaza $H_0$

### Aproximación mediante simulaciones
Bajo la hipótesis nula, los datos provienen de una distribución $X \sim Bin(8,\hat p)$, donde $\hat p$ es el valor estimado a partir de la muestra observada.

Se realizaran $\text{N\_SIM} = 10000$ simulaciones independientes, donde en cada simulación:
1. Se genera una muestra de tamaño $m=18$ (misma cantidad que los datos obtenidos) de una distribución $Bin(n=8,\hat{p})$.
2. Se estima el parámetro $p^{(j)}$ a partir de la muestra generada.
$$ \hat p^{(j)} = \frac{\bar X^{(j)}(m)}{8}. $$
3. Se calculan las probabilidades teóricas.
4. Se calcula el estadístico de Pearson $t_{(j)}$.
5. Aproximamos el p-valor:
$$
p\text{-valor} \approx \frac{\#\{j:t_{(j)}\ge t_{obs}\}}{N_{sim}}.
$$

In [3]:
N_SIM = 10_000

contador = 0
for _ in range(N_SIM):
    muestra_gen = np.random.binomial(n, p_est, m)
    p_j_est = muestra_gen.mean() / n
    
    frecuencias = np.bincount(muestra_gen, minlength=(n + 1))
    probabilidades = [binom.pmf(k, n, p_j_est) for k in range(n + 1)]

    total = sum(frecuencias)
    t_j = est_T(frecuencias, probabilidades, total, K)
    if t_j >= t_obs:
        contador += 1

p_valor = contador / N_SIM
print("p_valor simulado: ", p_valor)

p_valor simulado:  0.0164


Como *p-valor* es muy pequeño, se rechaza $H_0$